# Crypto Portfolio Optimization V5 (Bitget) — Walk-forward

Cách dùng: **chạy cell thư viện → chỉnh 2 cell PARAMS → chạy tuần tự các cell còn lại.** Mọi tham số backtest nằm trong 2 cell PARAMS.

Khác V3 (V3 giữ nguyên, không sửa):
- **Walk-forward**: tại mỗi candle `t`, fit trên `n_window` candle liền trước (không gồm `t`) → weight áp cho return của candle `t`. Tải `display_candles + n_window` nến: nửa đầu chỉ làm cửa sổ fit, nửa sau là phần backtest hiển thị.
- **Hard-rule trễ dữ liệu 1 ngày** (`data_lag_days`): hôm nay là T chỉ có data đến T-1, weight tối ưu từ đó chỉ ảnh hưởng từ T+1.
- Không `bfill` (chặn look-ahead); coin chưa đủ dữ liệu trong cửa sổ bị loại khỏi cửa sổ đó.
- Chọn tài sản / tối ưu / xếp hạng chỉ dùng dữ liệu cửa sổ. Grid `max_assets` vẫn có nhưng mỗi giá trị được walk-forward riêng.
- `mu` arithmetic (không ghép lãi theo giờ) và co về trung bình chéo (`mu_shrink`).
- Một `RF` cho optimizer + metrics; có phí giao dịch trên turnover.

Khác V4 (V4 giữ nguyên, không sửa) — **dữ liệu và thực thi theo Bitget UTA (API v3 public)**:
- Giá lấy từ `history-candles` của Bitget (tối đa 100 nến/request, cửa sổ ≤ 90 ngày, ~5 request/s) và cache CSV theo mã (`bitget_data/`), lần sau chỉ tải phần nến còn thiếu.
- Vũ trụ: `get_top_coins` xếp hạng spot USDT theo `turnover24h` (thanh khoản) của Bitget thay cho vốn hóa CoinGecko; lọc bằng `launchTime` trong `instruments`.
- `EURUSDT` không có trên Bitget: giá EUR = 1 / `USDTEUR` (đảo cả High/Low).
- Lệnh tối thiểu lấy theo từng mã từ `minOrderAmount` (1-20 USDT tùy mã), không còn một mức chung.

In [ ]:
!pip install --upgrade PyPortfolioOpt -q

In [ ]:
import time
start_time = time.time()

import os
import glob
import requests
import threading
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from math import ceil
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from rich.progress import Progress
from rich.console import Console
from typing import List
from pypfopt import EfficientFrontier, CovarianceShrinkage, risk_models

# Dữ liệu Bitget UTA (API v3, public): candles + instruments
BG = "https://api.bitget.com/api/v3/market"
BG_SYMBOL = {"EURUSDT": "USDTEUR"}       # EURUSDT không có trên Bitget -> giá EUR = 1 / USDTEUR (High/Low đảo chiều)
BG_COLS = ["open_time", "Open", "High", "Low", "Close", "volume", "quote_volume"]
BG_GAP = 0.21                            # giây giữa 2 request (~4.7 req/s; vượt ~5 req/s là bị 429)
_bg_lock, _bg_next, _INST = threading.Lock(), [0.0], [None]

def bg_interval(tf: str) -> str:
    n, u = int(tf[:-1]), tf[-1].lower()
    return f"{n}{u}" if u == "m" else f"{n}{u.upper()}utc" if u == "d" else f"{n}{u.upper()}"      # 15m, 1H, 4H, 1Dutc (nến ngày mở 00:00 UTC; "1D" thuần mở 16:00 UTC), 1W

def bg_get(path, **params):
    """GET public có giới hạn tốc độ dùng chung cho mọi luồng; 429 -> lùi rồi thử lại."""
    for attempt in range(6):
        with _bg_lock:
            slot = max(time.time(), _bg_next[0])
            _bg_next[0] = slot + BG_GAP
        time.sleep(max(0.0, slot - time.time()))
        r = requests.get(BG + path, params=params, timeout=20)
        if r.status_code == 429:
            time.sleep(1 + attempt)
            continue
        r.raise_for_status()
        j = r.json()
        if j["code"] != "00000":
            raise RuntimeError(f"Bitget {path} {params}: {j['code']} {j['msg']}")
        return j["data"] or []
    raise RuntimeError(f"Bitget {path}: bị 429 liên tục")

def bitget_instruments(refresh=False) -> pd.DataFrame:
    """Bảng instruments spot (index = symbol): launchTime, minOrderAmount, symbolType, status..."""
    if refresh or _INST[0] is None:
        _INST[0] = pd.DataFrame(bg_get("/instruments", category="SPOT")).set_index("symbol")
    return _INST[0]

def fetch_candles(sym, tf, start_ms, end_ms):
    """Tải các nến có open_time trong [start_ms, end_ms] theo cửa sổ cố định (≤100 nến, < 90 ngày), trả list dòng thô.
    Bitget trả nến có open >= startTime VÀ close (= open + step) <= endTime; startTime/endTime phải căn đúng lưới nến."""
    step = int(tf_delta(tf).total_seconds() * 1000)
    n = min(100, (90 * 86400000 - 1) // step)
    cur = -(-start_ms // step) * step                       # open đầu tiên trên lưới >= start_ms
    last = (end_ms + 1) // step * step - step               # open cuối cùng trên lưới <= end_ms
    rows = []
    while cur <= last:
        rows += bg_get("/history-candles", category="SPOT", symbol=sym, interval=bg_interval(tf), limit=n,
                       startTime=cur, endTime=min(last + step, cur + n * step))
        cur += n * step
    return rows

def update_candles(sym, tf, start_ms, end_ms, data_dir, launch_ms=0):
    """Cache CSV 1 file / (mã, khung). Chỉ tải phần đầu/đuôi còn thiếu so với [start_ms, end_ms] (đầu kỳ cắt theo launchTime)."""
    path = os.path.join(data_dir, f"{sym}_{tf}.csv")
    old = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame(columns=BG_COLS)
    step = int(tf_delta(tf).total_seconds() * 1000)
    lo = max(start_ms, launch_ms - launch_ms % step)
    if old.empty:
        spans = [(lo, end_ms)]
    else:
        first, last = int(old.open_time.min()), int(old.open_time.max())
        spans = [(lo, first - step)] * (lo < first) + [(last + step, end_ms)]
    rows = [r for a, b in spans if a <= b for r in fetch_candles(sym, tf, a, b)]
    if not rows:
        return 0
    new = pd.DataFrame(rows, columns=BG_COLS).astype(float)
    new["open_time"] = new["open_time"].astype("int64")
    pd.concat([old, new]).drop_duplicates("open_time").sort_values("open_time").to_csv(path, index=False)
    return len(rows)

def sync_candles(tickers, tf, start, end, data_dir, workers=4):
    """Đồng bộ cache candles của mọi ticker cho khoảng [start, end] (chuỗi YYYY-MM-DD, cả ngày end)."""
    os.makedirs(data_dir, exist_ok=True)
    s_ms = int(pd.Timestamp(start, tz="UTC").timestamp() * 1000)
    e_ms = int((pd.Timestamp(end, tz="UTC") + pd.Timedelta(days=1)).timestamp() * 1000) - 1
    inst = bitget_instruments()
    def one(t):
        sym = BG_SYMBOL.get(t, t)
        got = update_candles(sym, tf, s_ms, e_ms, data_dir, int(inst.loc[sym, "launchTime"]))
        print(f"  {t} {tf}: +{got} nến")
    with ThreadPoolExecutor(workers) as ex:
        list(ex.map(one, tickers))

DATA_VARS = ("price_df", "dd_hl", "MIN_ORDER")                         # sinh ra ở cell tải dữ liệu
RESULT_VARS = ("returns_wf", "weights_hist", "summary")   # sinh ra ở cell walk-forward

def reset_state(*names):
    """Xóa biến của các bước phía sau khi cell tham số/dữ liệu được chạy lại -> chạy lẻ cell sau sẽ báo NameError
    thay vì âm thầm dùng kết quả của lần chạy trước (không khớp tham số mới)."""
    for n in names:
        globals().pop(n, None)

def get_top_coins(n_coins: int = 10, quote: str = "USDT") -> list:
    """
    Top N coin spot trên Bitget (cặp {coin}{quote}) xếp theo turnover24h (thanh khoản, KHÔNG phải vốn hóa),
    chỉ lấy symbolType=crypto đang online, bỏ stablecoin / token wrapped / token không mong muốn trong blacklist.
    Trả List[str] symbol gốc viết hoa (vd 'BTC'), không trùng lặp.
    """
    blacklist = {
        "USDT", "USDC", "DAI", "TUSD", "UST", "WBTC", "STETH", "FIGR_HELOC", "WBT", "WSTETH",
        "WBETH", "USDS", "LEO", "BSC-USD", "WETH", "FDUSD", "USDP", "GUSD",
        "WEETH", "XMR", "USDE", "CBBTC", "SUSDS", "CRO", "USDT0", "PYUSD", "SUSDE", "HASH",
        "MNT", "USD1", "CC", "BGB", "OKB", "M", "USDF", "BUIDL", "JITOSOL",
        "PI", "RAIN", "JLP", "HTX", "SYRUPUSDC", "USYC", "USDG", "JSOL",
        "KCS", "RLUSD", "RSETH", "BNSOL", "WBNB", "BFUSD", "RIVER", "GT", "USDD", "RETH",
        "MYX", "KAS", "LBTC", "FBTC", "USDTB", "USTB", "USDC.E", "FLR", "SYRUPUSDT", "METH",
        "WM", "JAAA", "LSETH", "KHYPE", "USDY", "OUSG", "XDC", "SOLVBTC", "EUTBL", "USDAI",
        "COCA", "CLBTC", "BDX", "WFLR", "USD0", "JTRSY", "OSETH", "A7A5", "GHO", "EZETH",
        "IP", "JUPSOL", "STABLE", "FTN", "TBTC", "CTM", "EURC", "USR", "PIPPIN", "KAU",
        "YLDS", "SIREN", "USDGO"
    }
    inst = bitget_instruments(refresh=True)
    try:
        tk = pd.DataFrame(bg_get("/tickers", category="SPOT")).set_index("symbol")
    except Exception as e:
        print(f"❌ Lỗi khi gọi tickers Bitget: {e}")
        return []
    ok = inst[(inst.symbolType == "crypto") & (inst.quoteCoin == quote) & (inst.status == "online") & ~inst.baseCoin.str.upper().isin(blacklist)]
    turn = pd.to_numeric(tk["turnover24h"], errors="coerce").reindex(ok.index).fillna(0).sort_values(ascending=False)
    return list(dict.fromkeys(ok.loc[turn.index, "baseCoin"].str.upper()))[:n_coins]

def get_start_date(
    timeframe: str,
    required_samples: int = 10000,
    end_date: datetime | None = None,
) -> tuple[datetime, int]:
    """
    Calculate start date to obtain at least `required_samples` candles.

    Parameters
    ----------
    timeframe : str
        e.g. '1m', '5m', '15m', '1h', '4h', '1d', '1w'
    required_samples : int
        Minimum number of candles required.
    end_date : datetime
        End date. Defaults to current UTC time.

    Returns
    -------
    start_date : datetime
    required_days : int
    """
    if end_date is None:
        end_date = datetime.utcnow()

    unit = timeframe[-1].lower()
    value = int(timeframe[:-1])

    if unit == "m":
        candles_per_day = 24 * 60 / value

    elif unit == "h":
        candles_per_day = 24 / value

    elif unit == "d":
        candles_per_day = 1 / value

    elif unit == "w":
        candles_per_day = 1 / (7 * value)

    else:
        raise ValueError(f"Unsupported timeframe: {timeframe}")

    required_days = ceil(required_samples / candles_per_day)
    start_date = end_date - timedelta(days=required_days)

    return start_date, required_days

import warnings
from joblib import Parallel, delayed, effective_n_jobs

# ====== Thư viện V4: walk-forward, không look-ahead (RF lấy từ block params) ======
def tf_delta(tf: str) -> pd.Timedelta:
    return pd.Timedelta(tf.replace('m', 'min'))


def periods_per_year(tf: str) -> float:
    return 365 * 24 * 3600 / tf_delta(tf).total_seconds()


def load_prices(tickers, start, end, timeframe, data_dir="bitget_data", col="Close"):
    """Giá `col` của mọi ticker trên 1 index chung (đọc cache CSV). Chỉ ffill (causal), KHÔNG bfill:
    coin niêm yết muộn giữ NaN ở đoạn trước khi có giá. Ticker trong BG_SYMBOL (EURUSDT) đảo giá: 1/x, High <-> Low."""
    delta = tf_delta(timeframe)
    idx = pd.date_range(pd.Timestamp(start, tz="UTC"),
                        pd.Timestamp(end, tz="UTC") + pd.Timedelta(days=1) - delta, freq=delta)
    def load_one(t):
        sym = BG_SYMBOL.get(t, t)
        path = os.path.join(data_dir, f"{sym}_{timeframe}.csv")
        if not os.path.exists(path):
            print(f"⚠️  Bỏ {t}: không có file")
            return None
        df = pd.read_csv(path)
        inv = t in BG_SYMBOL
        s = df.set_index(pd.to_datetime(df["open_time"], unit="ms", utc=True))[{"High": "Low", "Low": "High"}.get(col, col) if inv else col].dropna()
        s = (1 / s.where(s > 0)) if inv else s
        s = s[~s.index.duplicated()]
        return s[s > 0].reindex(idx)

    # ponytail: tuần tự; mỗi mã chỉ 1 file CSV nên đủ nhanh
    cols = {t: s for t, s in zip(tickers, map(load_one, tickers)) if s is not None}
    return pd.DataFrame(cols).ffill()


def cov_matrix(win, P, model, span):
    shrink = lambda: CovarianceShrinkage(win, frequency=P).ledoit_wolf()
    if model == "shrinkage":
        return shrink()
    if model == "sample":
        return risk_models.sample_cov(win, frequency=P)
    S = risk_models.exp_cov(win, frequency=P, span=span)
    return S if model == "exp" else 0.5 * S + 0.5 * shrink()


def max_drawdown(r: pd.Series) -> float:
    cum = (1 + r).cumprod()
    return (cum / cum.cummax() - 1).min()


def optimize_window(win, rets, P, cfg, max_assets):
    """Chọn tài sản + tối ưu CHỈ trên `win` (dữ liệu quá khứ). Trả {strategy: weights}."""
    mu = rets.mean() * P                      # arithmetic, không ghép lãi theo giờ
    sd = rets.std() * np.sqrt(P)
    ranked = (mu / sd).sort_values(ascending=False).index

    req = {a: m for a, m in cfg["required_assets"].items() if a in win.columns}
    sel = list(dict.fromkeys(list(req) + list(ranked)))[:max_assets]
    lo = [req.get(a, cfg["min_weight"]) for a in sel]
    hi = [cfg["eur_max_weight"] if a in cfg["eur_assets"]
          else 1.0 if a in req else cfg["other_max_weight"] for a in sel]   # mã không bắt buộc bị chặn trần
    if sum(lo) > 1 or sum(hi) < 1 or any(l > h for l, h in zip(lo, hi)):    # không khả thi -> bỏ bước này
        return {}

    s = cfg["mu_shrink"]                      # co mu về trung bình chéo -> bớt nhiễu
    mu_s = (1 - s) * mu[sel] + s * mu[sel].mean()
    S = cov_matrix(win[sel], P, cfg["risk_model"], cfg["span"])
    bounds = list(zip(lo, hi))

    def solve(kind):
        ef = EfficientFrontier(mu_s, S, weight_bounds=bounds, solver_options={"adaptive_rho_interval": 25})
        try:
            if kind == "sharpe":
                ef.max_sharpe(risk_free_rate=RF)
            elif kind == "optimal":
                ef.efficient_return(cfg["target_return"])
            else:
                ef.min_volatility()
        except Exception:                     # vd: mọi mu < RF, target vượt khả thi
            ef = EfficientFrontier(mu_s, S, weight_bounds=bounds, solver_options={"adaptive_rho_interval": 25})
            ef.min_volatility()
        return dict(ef.clean_weights())

    return {k: solve(k) for k in ("sharpe", "volatility", "optimal")}


def round_weights(w, step):
    """Làm tròn weight về bội số của `step` (largest remainder): tổng giữ nguyên, không có số lẻ hơn `step`.
    Sàn/trần là bội của step nên vẫn được giữ."""
    if not step or not w:
        return w
    names = list(w)
    x = np.array([w[n] for n in names]) / step
    units = int(round(x.sum()))
    base = np.floor(x + 1e-9).astype(int)
    for i in np.argsort(-(x - base))[:max(0, units - base.sum())]:
        base[i] += 1
    return {n: float(b * step) for n, b in zip(names, base)}


def make_targets(win, P, cfg, max_assets_list, mdd_hist=None):
    """Target weight tại 1 mốc rebalance. Benchmark không phụ thuộc max_assets -> tính 1 lần;
    opt_*_m{m} tính riêng cho từng m trong max_assets_list."""
    rets = win.pct_change().dropna()
    cols = list(rets.columns)
    out = {"equal_weight": {c: 1 / len(cols) for c in cols}}

    inv = 1 / rets.std().replace(0, np.nan).dropna()
    out["risk_parity"] = (inv / inv.sum()).to_dict()

    # equal absolute drawdown: phần dư (cash_asset) không tham gia ngân sách rủi ro
    cash = cfg["cash_asset"]
    mdds = {c: abs(max_drawdown(rets[c])) for c in cols if c != cash}
    if mdd_hist:                              # MDD lịch sử dài (vd 5 năm) thay cho MDD của cửa sổ fit
        mdds = {c: mdd_hist.get(c, m) for c, m in mdds.items()}
    mdds = {c: m for c, m in mdds.items() if m > 0}
    for tgt in map(abs, cfg["mdd_targets"]):
        if not mdds:
            break
        w = {c: (tgt / len(mdds)) / m for c, m in mdds.items()}
        tot = sum(w.values())
        if tot > 1:
            w = {c: x / tot for c, x in w.items()}
            tot = 1.0
        if cash in cols:
            w[cash] = 1 - tot                 # nếu thiếu cash_asset: phần dư để tiền mặt (0%)
        out[f"absolute_equal_dd_{int(tgt * 100)}"] = w

    for m in max_assets_list:
        try:
            out.update({f"opt_{k}_m{m}": v for k, v in optimize_window(win, rets, P, cfg, m).items()})
        except Exception as e:
            print(f"  ⚠️ optimize lỗi (m={m}), giữ weight cũ: {e}")
    return out


def targets_chunk(prices, ts, n_window, timeframe, cfg, max_assets_list, lag, dd_hl):
    """Pha 1 (song song): target của mọi strategy tại các mốc rebalance `ts`. Chỉ phụ thuộc data cửa sổ, không phụ thuộc đường đi."""
    # cvxpy hay báo "Solution may be inaccurate" ở các bước tối ưu biên; kết quả vẫn dùng được -> tắt cho gọn
    warnings.filterwarnings("ignore", message="Solution may be inaccurate", category=UserWarning)
    P = periods_per_year(timeframe)
    out = []
    for t in ts:
        mdd_hist = None
        if dd_hl is not None:                     # MDD từng mã trong N năm gần nhất, chỉ dùng data đã biết
            end = prices.index[t - lag - 1] + tf_delta(timeframe)
            dh, dl = dd_hl                        # High, Low khung mdd_timeframe: đỉnh = cummax(High), đáy = Low
            m = (dh.index + tf_delta(cfg["mdd_timeframe"]) <= end) & (dh.index > end - pd.DateOffset(years=cfg["mdd_years"]))
            dh, dl = dh[m], dl[m]
            mdd_hist = {c: abs((dl[c][dh[c].notna()] / dh[c].dropna().cummax() - 1).min())
                        for c in dh.columns if dh[c].notna().sum() > 1}
        win = prices.iloc[t - lag - n_window:t - lag].dropna(axis=1)
        tg = make_targets(win, P, cfg, max_assets_list, mdd_hist)
        out.append({n: round_weights(w, cfg["weight_step"]) for n, w in tg.items()})
    return out


def limit_move(prev, tgt, cap, step):
    """Dịch weight từ `prev` về `tgt` (cả hai nằm trên lưới `step`, tổng 1) sao cho mỗi mã đổi tối đa `cap` (chuyển theo cặp tăng <-> giảm
    nên tổng giữ nguyên). Mỗi mã đi đơn điệu từ prev tới tgt nên sàn/trần weight vẫn được giữ. Mã chưa chuyển hết thì tiếp ở lần rebalance sau."""
    r = np.rint((tgt - prev) / step).astype(int)            # cần đổi (đơn vị step), có dấu
    a = np.full(len(r), max(1, int(round(cap / step))))     # hạn mức còn lại mỗi mã
    out = np.zeros(len(r), int)
    while True:
        pos, neg = np.flatnonzero((r > 0) & (a > 0)), np.flatnonzero((r < 0) & (a > 0))
        if not len(pos) or not len(neg):
            break
        i, j = pos[np.argmax(r[pos])], neg[np.argmin(r[neg])]
        m = min(r[i], -r[j], a[i], a[j])
        out[i] += m; out[j] -= m; r[i] -= m; r[j] += m; a[i] -= m; a[j] -= m
    return np.round(prev + out * step, 10)


def walk_forward(prices, n_window, timeframe, cfg, max_assets_list, rebalance_every=1, fee=0.001, threshold=0.0, lag=0, dd_hl=None, n_jobs=-1, max_move=None):
    """Tại candle t: fit trên prices[t-lag-n_window : t-lag] -> áp weight cho return của candle t.
    `lag` (candle) = độ trễ dữ liệu: tại thời điểm t chỉ biết giá đến t-lag (hard-rule: dữ liệu luôn trễ 1 ngày).
    Giữa 2 lần rebalance weight trôi theo giá. Phí = fee * turnover.
    Tại mỗi mốc rebalance, strategy chỉ giao dịch nếu weight lệch >= threshold (max |target - hiện tại|).
    `max_move` = mức đổi weight tối đa của MỖI mã giữa 2 lần rebalance (vd 0.01 = 1%); lớn hơn thì dịch dần về target qua các lần sau.
    Trả (returns, weights_hist = weight THỰC HIỆN tại các lần giao dịch, n_target = số mã tối đa trong TARGET của từng strategy).
    Pha 1: tính target mọi mốc rebalance song song (benchmark + mọi max_assets dùng chung cửa sổ/MDD).
    Pha 2: vòng candle tuần tự, vector hóa mọi strategy thành ma trận (S x A)."""
    cols = list(prices.columns)
    pos = {c: i for i, c in enumerate(cols)}
    start = n_window + lag
    total = len(prices) - start
    ts = list(range(start, len(prices), rebalance_every))

    t0 = time.time()
    chunks = np.array_split(ts, max(1, min(len(ts), 4 * effective_n_jobs(n_jobs))))
    res = Parallel(n_jobs=n_jobs)(delayed(targets_chunk)(prices, c.tolist(), n_window, timeframe, cfg, max_assets_list, lag, dd_hl) for c in chunks)
    targets_list = [x for r in res for x in r]
    print(f"  pha 1: {len(ts)} mốc rebalance x {len(max_assets_list)} giá trị max_assets - {time.time() - t0:.0f}s")

    hold = "hold_" + cfg["hold_asset"]
    names = list(dict.fromkeys([n for tg in targets_list for n in tg] + ([hold] if cfg["hold_asset"] in pos else [])))
    row = {n: i for i, n in enumerate(names)}
    S, A = len(names), len(cols)
    Wm = np.zeros((S, A))                         # weight hiện tại (đã trôi theo giá) của từng strategy
    live = np.zeros(S, bool)                      # strategy đã có weight lần đầu chưa
    Lm = np.zeros((S, A))                         # weight thực hiện ở lần giao dịch gần nhất (trên lưới weight_step, không trôi theo giá)
    n_target = {}
    if max_move and not cfg["weight_step"]:
        raise ValueError("max_move cần cfg['weight_step'] > 0 (weight nằm trên lưới)")
    Rm = np.full((total, S), np.nan)
    rets = prices.pct_change().fillna(0).to_numpy()
    hist = {}
    targets_it = iter(targets_list)

    for k, t in enumerate(range(start, len(prices))):
        turn = np.zeros(S)
        if k % rebalance_every == 0:
            targets = next(targets_it)
            if k == 0 and cfg["hold_asset"] in pos:
                targets[hold] = {cfg["hold_asset"]: 1.0}   # buy & hold, không rebalance
            for name, tw in targets.items():
                i = row[name]
                vec = np.zeros(A)
                for a, x in tw.items():
                    vec[pos[a]] = x
                if live[i] and np.abs(vec - Wm[i]).max() < threshold:
                    continue                      # lệch chưa đủ -> giữ nguyên, không phí
                n_target[name] = max(n_target.get(name, 0), int((vec > 0).sum()))
                if max_move and live[i]:
                    vec = limit_move(Lm[i], vec, max_move, cfg["weight_step"])   # mỗi mã chỉ đổi tối đa max_move mỗi lần rebalance
                turn[i] = np.abs(vec - Wm[i]).sum()
                Wm[i] = Lm[i] = vec
                live[i] = True
                hist.setdefault(name, {})[prices.index[t]] = pd.Series(vec, index=cols)
        r = rets[t]
        pr = Wm @ r
        Rm[k] = np.where(live, pr - fee * turn, np.nan)
        Wm *= 1 + r
        Wm /= (1 + pr)[:, None]

    returns = pd.DataFrame(Rm, index=prices.index[start:], columns=names).dropna(axis=1, how="all")
    return returns, {n: pd.DataFrame(h).T.fillna(0) for n, h in hist.items()}, n_target


def summarize(returns, timeframe, weights_hist):
    P = periods_per_year(timeframe)
    rows = {}
    for name, r in returns.items():
        eq = (1 + r).cumprod()
        sd = r.std()
        dd = np.sqrt((np.minimum(r - RF / P, 0) ** 2).mean())      # downside deviation, ngưỡng = RF/P (cùng ngưỡng với Sharpe)
        wh = weights_hist.get(name)
        # turnover/năm = tổng |Δweight| giữa các lần rebalance (không tính trôi giá)
        tn = wh.diff().abs().sum().sum() if wh is not None and len(wh) > 1 else 0.0
        rows[name] = {
            "return_cagr": eq.iloc[-1] ** (P / len(r)) - 1,
            "volatility": sd * np.sqrt(P),
            "sharpe": (r.mean() - RF / P) / sd * np.sqrt(P) if sd > 0 else 0.0,
            "sortino": (r.mean() - RF / P) / dd * np.sqrt(P) if dd > 0 else 0.0,
            "max_drawdown": (eq / eq.cummax() - 1).min(),
            "turnover_per_year": tn * P / len(r),
        }
    return pd.DataFrame(rows).T.sort_values("sortino", ascending=False)


def filter_listed(symbols, quote, start_date, require_full_history=True):
    """Giữ (theo thứ tự đầu vào) các symbol có cặp spot {symbol}{quote} online trên Bitget và, nếu
    require_full_history, có launchTime <= start_date (đủ lịch sử cho cả cửa sổ fit + backtest).
    Lọc TRƯỚC khi tải/rolling để mã niêm yết muộn không bị loại âm thầm ở từng cửa sổ. Không gọi mạng (dùng instruments)."""
    inst = bitget_instruments()
    kept = []
    for sym in symbols:
        pair = sym + quote
        if pair not in inst.index or inst.loc[pair, "status"] != "online":
            print(f"  ✗ {sym}: không có cặp {pair} online trên Bitget spot")
            continue
        listed = pd.to_datetime(int(inst.loc[pair, "launchTime"]), unit="ms").date()
        if require_full_history and listed > start_date:
            print(f"  ✗ {sym}: niêm yết {listed} > {start_date} (thiếu lịch sử)")
        else:
            kept.append(sym)
    return kept


In [ ]:
# ====================== PARAMS 1/2: DỮ LIỆU & BACKTEST ======================
reset_state(*DATA_VARS, *RESULT_VARS)       # đổi tham số -> dữ liệu và kết quả cũ không còn hợp lệ, phải chạy lại các cell sau

# --- Khung thời gian & độ dài backtest ---
timeframe = '1h'            # Khung nến Bitget: 1m 5m 15m 30m 1h 4h 6h 12h 1d 1w
display_candles = 10000     # Số nến backtest được hiển thị/đánh giá (phần out-of-sample)
n_window = display_candles  # Số nến lịch sử liền trước dùng để fit weight ở mỗi bước.
                            # Tổng nến tải về = display_candles + n_window (mặc định gấp đôi)

# --- Tần suất & chi phí ---
n_jobs = -1                 # Số tiến trình chạy song song tính target (mỗi mốc rebalance chạy độc lập, gồm cả grid max_assets).
                            # -1 = dùng mọi core, 1 = tuần tự, 4 = tối đa 4 core
rebalance_days = 1          # Chu kỳ tính lại weight, tính bằng NGÀY (30 = hàng tháng, 7 = hàng tuần, 1 = hàng ngày).
                            # Tự quy ra số candle theo timeframe (1h -> 720, 1d -> 30). Giữa 2 lần
                            # rebalance weight trôi theo giá. Nhỏ hơn 1 candle -> mỗi candle (chậm nhất)
rebalance_threshold = 0.01  # Chỉ rebalance khi weight lệch >= ngưỡng này (điểm % tuyệt đối của bất kỳ asset nào,
                            # 0.01 = 1%) so với target mới, kiểm tra tại mỗi mốc rebalance_days.
                            # 0 = luôn rebalance; chưa đủ lệch -> giữ nguyên, không tốn phí
data_lag_days = 1           # HARD-RULE độ trễ dữ liệu (ngày): hôm nay là T chỉ có data đến T-1, nên weight tối ưu
                            # từ data đó chỉ ảnh hưởng từ T+1. Quy ra candle theo timeframe. 0 = tắt (chỉ để so sánh)
max_daily_weight_change = 0.01   # Mỗi mã chỉ được tăng/giảm tối đa chừng này weight MỖI NGÀY (0.01 = 1%): target xa hơn thì dịch dần qua các ngày,
                            # tránh sốc 10-20% trong 1 ngày. Tính theo chu kỳ rebalance (rebalance_days ngày -> hạn mức nhân tương ứng).
                            # Lần phân bổ đầu tiên từ tiền mặt không bị giới hạn. 0 = tắt. Cần cfg['weight_step'] > 0
fee = 0.001                 # Phí mỗi lần giao dịch, tính trên turnover (0.001 = 0.1% - mức spot mặc định của Bitget, CHƯA đối chiếu với fee-rate tài khoản)
RF = 0.03                   # Lãi suất phi rủi ro / năm, dùng cho max-Sharpe VÀ cho chỉ số Sharpe

# --- Vũ trụ tài sản ---
top_n_coins = 8             # Lấy top N coin spot Bitget theo turnover24h (thanh khoản; đã lọc stablecoin, token wrapped)
candidate_pool = 30         # Lấy top N này theo thanh khoản rồi LỌC (bỏ mã không có cặp spot online trên Bitget hoặc thiếu lịch sử)
                            # cho đến khi đủ top_n_coins. Phải >= top_n_coins; tăng nếu nhiều mã bị loại
require_full_history = True # True = chỉ giữ mã đã niêm yết trước ngày bắt đầu dữ liệu (đủ cho cửa sổ fit + trễ + backtest)
quote_asset = 'USDT'        # Đồng định giá; ticker = symbol + quote_asset (vd BTC -> BTCUSDT)
extra_tickers = ['EURUSDT', 'PAXGUSDT']   # Tài sản thêm tay, luôn nằm trong vũ trụ (EUR = 1/USDTEUR, vàng)

# --- Tải dữ liệu ---
data_dir = 'bitget_data'    # Thư mục cache CSV Bitget (gitignore)
download_workers = 4        # Số luồng tải song song (tốc độ chung bị giới hạn ~5 request/s bởi BG_GAP)

# --- Dẫn xuất (không cần sửa) ---
max_move = max_daily_weight_change * rebalance_days if max_daily_weight_change else None      # hạn mức đổi weight mỗi mã / lần rebalance
rebalance_every = max(1, round(rebalance_days * 86400 / tf_delta(timeframe).total_seconds()))   # số candle
lag_candles = round(data_lag_days * 86400 / tf_delta(timeframe).total_seconds())              # độ trễ, số candle

In [ ]:
# ====================== PARAMS 2/2: CHIẾN LƯỢC ======================
reset_state(*RESULT_VARS)                   # đổi tham số chiến lược -> kết quả walk-forward cũ không còn hợp lệ

# Grid test: chạy walk-forward riêng cho từng giá trị max_assets rồi so sánh.
# Chậm theo số giá trị (range(2, 11) = 9 lần). Thu hẹp (vd range(4, 7)) khi thử nhanh.
max_assets_range = range(2, 11)
max_assets_limit = 10       # Trần số assets trong port: bỏ giá trị m lớn hơn trần, và bỏ mọi strategy có lúc nắm > trần mã
                            # (vd equal_weight trên cả vũ trụ) khỏi kết quả -> không bao giờ được chọn làm "best"

cfg = dict(
    # --- Ràng buộc trọng số ---
    required_assets={'EURUSDT': 0.15, 'PAXGUSDT': 0.09, 'BTCUSDT': 0.06, 'ETHUSDT': 0.04},
                            # Mã bắt buộc có mặt + trọng số TỐI THIỂU. Tổng sàn phải <= 100%
    min_weight=0.01,        # Trọng số tối thiểu của các mã còn lại được chọn
    other_max_weight=0.10,  # Trọng số TỐI ĐA của mã không nằm trong required_assets (áp cho các strategy opt_*).
                            # Nếu tổng trần không đủ 100% (quá ít mã bắt buộc) thì bước tối ưu đó bị bỏ qua
    eur_assets={'EURUSDT', 'EURIUSDT'},   # Nhóm tài sản bị chặn trần bởi eur_max_weight
    eur_max_weight=0.34,    # Trọng số tối đa cho mỗi mã trong eur_assets

    # --- Mục tiêu tối ưu ---
    target_return=0.05,     # Return/năm mục tiêu cho strategy 'opt_optimal' (efficient_return).
                            # Không đạt được -> tự chuyển về min-volatility

    # --- Ước lượng đầu vào cho optimizer ---
    risk_model='exp_shrink',  # Ma trận hiệp phương sai: 'shrinkage' (Ledoit-Wolf) | 'sample' |
                              # 'exp' (EWMA) | 'exp_shrink' (50% EWMA + 50% Ledoit-Wolf)
    cov_span_days=60,       # Span EWMA tính theo ngày (chỉ dùng cho 'exp'/'exp_shrink')
    mu_shrink=0.5,          # Co return kỳ vọng về trung bình chéo: 0 = giữ nguyên mu thô,
                            # 1 = mọi mã cùng mu. Tăng để bớt nhiễu / bớt đuổi theo mã thắng

    # --- Làm tròn weight ---
    weight_step=0.01,       # Mọi target weight là bội số của bước này (0.01 = 1%, không số lẻ hơn). 0 = tắt

    # --- Benchmark ---
    mdd_targets=(-0.08, -0.13, -0.21, -0.34, -0.55),
                            # MDD mục tiêu cho các danh mục absolute_equal_dd_xx
    mdd_years=5,            # absolute_equal_dd lấy MDD từng mã trong N năm gần nhất (rolling, chỉ data đã biết)
                            # thay vì chỉ trong cửa sổ fit; mã niêm yết ngắn hơn N năm dùng toàn bộ lịch sử có
    mdd_timeframe='1d',     # Khung nến dùng tính MDD dài hạn (tải riêng, nhẹ hơn nhiều so với nến 1h).
                            # MDD = (Low - cummax(High)) / cummax(High): bắt cả đợt sụt trong ngày
    cash_asset='EURUSDT',   # Mã nhận phần dư của absolute_equal_dd (không tham gia ngân sách rủi ro)
    hold_asset='BTCUSDT',   # Mã dùng làm benchmark buy & hold
)


In [ ]:
# --- Tải & nạp dữ liệu ---
t_load = time.time()
reset_state(*DATA_VARS, *RESULT_VARS)
end_date = date.today() - timedelta(days=1)
start_date, days = get_start_date(timeframe, display_candles + n_window + lag_candles + 24, end_date)   # +24 nến đệm
start_str, end_str = start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")
print(f"{timeframe} -> {days} days: {start_str} .. {end_str}")

# Lọc vũ trụ TRƯỚC khi tải/rolling: top theo thanh khoản nhưng phải có cặp spot online và đủ lịch sử
print(f"Lọc top {candidate_pool} ứng viên:")
top_coins = filter_listed(get_top_coins(candidate_pool, quote_asset), quote_asset, start_date, require_full_history)[:top_n_coins]
if len(top_coins) < top_n_coins:
    print(f"⚠️ Chỉ có {len(top_coins)}/{top_n_coins} mã đạt điều kiện, tăng candidate_pool hoặc giảm top_n_coins")
tickers = extra_tickers + [s + quote_asset for s in top_coins]
print("Tickers:", tickers)

sync_candles(tickers, timeframe, start_str, end_str, data_dir, download_workers)

price_df = load_prices(tickers, start_str, end_str, timeframe, data_dir)

# MDD dài hạn cho absolute_equal_dd: nến mdd_timeframe, cfg['mdd_years'] năm gần nhất
mdd_start = end_date - relativedelta(years=cfg['mdd_years'])
sync_candles(tickers, cfg['mdd_timeframe'], mdd_start.strftime("%Y-%m-%d"), end_str, data_dir, download_workers)
dd_hl = tuple(load_prices(tickers, mdd_start.strftime("%Y-%m-%d"), end_str, cfg['mdd_timeframe'], data_dir, col=k)
              for k in ("High", "Low"))               # đỉnh theo High, đáy theo Low -> bắt cả sụt trong ngày
print(f"MDD lookback: {len(dd_hl[0])} nến {cfg['mdd_timeframe']} ({cfg['mdd_years']} năm)")
price_df = price_df.iloc[-(display_candles + n_window + lag_candles):]   # cắt đúng số nến cần
n_fit = min(n_window, len(price_df) - 1 - lag_candles)      # n_window là input, n_fit là giá trị thực dùng -> chạy lại cell này không bị tích lũy
print(f"{len(price_df)} candles = cửa sổ fit {n_fit} + trễ {lag_candles} + backtest {len(price_df) - n_fit - lag_candles}")
print("Số giá hợp lệ / mã:"); print(price_df.notna().sum().to_string())
# Lệnh tối thiểu ($) theo mã: minOrderAmount của Bitget (tính bằng quote). EURUSDT = USDTEUR có quote là EUR -> quy ra USD theo giá EUR cuối
inst = bitget_instruments()
MIN_ORDER = {t: float(inst.loc[BG_SYMBOL.get(t, t), "minOrderAmount"]) * (float(price_df[t].dropna().iloc[-1]) if t in BG_SYMBOL else 1.0) for t in price_df.columns}
print("Min order ($):", {t: round(v, 2) for t, v in MIN_ORDER.items()})
print(f"⏱ Tải & nạp dữ liệu: {time.time() - t_load:.1f}s")

In [ ]:
# --- Chạy walk-forward (pha 1 song song theo mốc rebalance x grid max_assets, pha 2 vector hóa) + bảng kết quả ---
reset_state(*RESULT_VARS)
cfg['span'] = int(cfg['cov_span_days'] * 24 * 3600 / tf_delta(timeframe).total_seconds())     # span EWMA (candle)
grid = [m for m in max_assets_range if m <= max_assets_limit]
assert grid, f"max_assets_range {list(max_assets_range)} không có giá trị nào <= max_assets_limit={max_assets_limit}"
t_run = time.time()
returns_wf, weights_hist, n_target = walk_forward(price_df, n_fit, timeframe, cfg, grid, rebalance_every=rebalance_every,
                                                  fee=fee, threshold=rebalance_threshold, lag=lag_candles, dd_hl=dd_hl, n_jobs=n_jobs, max_move=max_move)
over = [n for n, k in n_target.items() if k > max_assets_limit]    # strategy có target nắm > trần mã (đang dịch dần có thể tạm giữ nhiều mã hơn)
returns_wf, weights_hist = returns_wf.drop(columns=over), {n: w for n, w in weights_hist.items() if n not in over}
if over:
    print(f"Bỏ {len(over)} strategy vượt trần {max_assets_limit} mã: {over}")
summary = summarize(returns_wf, timeframe, weights_hist)
print(f"⏱ Walk-forward (grid {grid}, n_jobs={n_jobs}): {time.time() - t_run:.1f}s")
(summary * [100, 100, 1, 1, 100, 1]).round(2)   # cột: return_cagr%, volatility%, sharpe, sortino, max_drawdown%, turnover/năm

In [ ]:
# --- Plot: equity của mọi strategy (net of fees) ---
fig = go.Figure()
for name, r in returns_wf.items():
    fig.add_trace(go.Scatter(x=r.index, y=(1 + r).cumprod(), mode="lines", name=name,
                             line=dict(width=3 if name.startswith(("opt_", "hold_")) else 1)))
fig.update_layout(title="Walk-forward equity (net of fees)", hovermode="x unified")
fig.show()

In [ ]:
# --- Strategy tốt nhất theo 1 chỉ số: weight hiện tại + 1 chart gộp (equity = đỉnh stack, weight = độ dày từng lớp) ---
def simulate_account(metric, init_balance, min_order=None, dca_amount=0.0, dca_days=30):
    """Mô phỏng tài khoản theo strategy tốt nhất `metric`. Mỗi mốc rebalance, lệnh có |target - hiện tại| < min order ($) bị bỏ; min order theo từng mã (MIN_ORDER từ Bitget), `min_order` là số thì áp 1 mức chung cho mọi mã.
    DCA: cứ mỗi `dca_days` ngày nạp thêm `dca_amount` ($) và rebalance ngay về target hiện hành (dca_amount = 0 -> tắt).
    Trả (tên strategy, giá trị $ từng mã + CASH theo thời gian, tổng phí, số lệnh khớp, số lệnh bị bỏ, tổng vốn đã nạp theo thời gian)."""
    best = summary[metric].idxmax()
    wh = weights_hist[best].reindex(columns=price_df.columns, fill_value=0.0)     # target weight tại các mốc rebalance
    idx = returns_wf[best].dropna().index
    r = price_df.pct_change().fillna(0).loc[idx].to_numpy()
    tw = dict(zip(idx.get_indexer(wh.index), wh.to_numpy()))                      # candle i -> target weight (áp trước return của candle i)
    mo = np.array([MIN_ORDER[c] for c in price_df.columns]) if min_order is None else np.full(r.shape[1], float(min_order))   # lệnh tối thiểu ($) từng mã
    hold, cash = np.zeros(r.shape[1]), init_balance                               # vị thế ($) từng mã, tiền mặt USDT (không sinh lãi)
    H, C, fees, done, skipped = np.empty(r.shape), np.empty(len(idx)), 0.0, 0, 0
    every = max(1, round(dca_days * 86400 / tf_delta(timeframe).total_seconds()))   # chu kỳ DCA, số candle
    dep = np.zeros(len(idx))
    if dca_amount > 0:
        dep[every::every] = dca_amount                                            # nạp đầu candle, trước return của candle đó
    target = None
    for i, ri in enumerate(r):
        cash += dep[i]
        target = tw.get(i, target)                                                # target mới nhất đã biết
        if i in tw or dep[i] > 0:
            d0 = target * (cash + hold.sum()) - hold                               # $ cần mua (+) / bán (-)
            d = np.where(np.abs(d0) < mo, 0.0, d0)
            sell = -np.minimum(d, 0)
            cash += sell.sum() * (1 - fee); fees += sell.sum() * fee              # bán trước để có tiền mua
            buy = np.maximum(d, 0)
            buy *= min(1.0, cash / (buy.sum() * (1 + fee))) if buy.sum() > 0 else 1.0   # không đủ tiền -> giảm đều
            buy[buy < mo] = 0.0
            cash -= buy.sum() * (1 + fee); fees += buy.sum() * fee
            hold += buy - sell
            done += int(((buy > 0) | (sell > 0)).sum()); skipped += int(((np.abs(d0) > 1e-9) & (buy == 0) & (sell == 0)).sum())
        hold *= 1 + ri
        H[i], C[i] = hold, cash
    values = pd.DataFrame(H, index=idx, columns=price_df.columns)
    values = values.loc[:, values.max() > 1e-9 * max(init_balance, 1)]            # bỏ mã chưa từng nắm giữ
    values["CASH"] = C
    return best, values, fees, done, skipped, pd.Series(init_balance + dep.cumsum(), index=idx)

def plot_stack(values, title):
    """Stacked area $ theo mã: đỉnh stack = equity, độ dày từng lớp = vị thế (weight). Đường đen = equity (hover hiện tổng)."""
    fig = go.Figure([go.Scatter(x=values.index, y=values[c], stackgroup="v", name=c, line=dict(width=0.5)) for c in values.columns])
    fig.add_trace(go.Scatter(x=values.index, y=values.sum(axis=1), name="Equity", line=dict(color="black", width=1.5)))
    return fig.update_layout(title=title, hovermode="x unified")

def plot_weights(wh, title):
    """Stacked area TARGET weight (%) của strategy, đúng giá trị đã làm tròn weight_step tại mỗi lần rebalance (giữ nguyên đến lần sau).
    Weight trôi theo giá nằm ở chart equity & weights."""
    wh = wh.loc[:, (wh > 0).any()] * 100
    end = returns_wf.index[-1]
    if wh.index[-1] < end:                    # lần giao dịch cuối có thể sớm hơn nến cuối backtest -> kéo dài weight đang giữ đến hết kỳ
        wh = pd.concat([wh, wh.iloc[[-1]].set_axis([end])])
    fig = go.Figure([go.Scatter(x=wh.index, y=wh[c], stackgroup="w", name=c, line=dict(width=0.5, shape="hv")) for c in wh.columns])
    return fig.update_layout(title=title, hovermode="x unified", yaxis=dict(ticksuffix="%", range=[0, 100]))

def show_best(metric):
    best = summary[metric].idxmax()
    print(f"Best {metric}: {best} ({summary.loc[best, metric]:.4g})")
    cur = weights_hist[best].iloc[-1]
    print("Weight hiện tại (lần rebalance gần nhất):")
    print((cur[cur > 0].sort_values(ascending=False) * 100).round(2).astype(str) + "%")
    _, values, *_ = simulate_account(metric, 1.0, 0.0)                            # vốn 1, không giới hạn lệnh tối thiểu
    plot_stack(values, f"Equity & weights - {best} (vốn = 1)").show()
    plot_weights(weights_hist[best], f"Target weights (làm tròn theo weight_step) - {best}").show()

show_best("return_cagr")

In [ ]:
show_best("sortino")

In [ ]:
# --- Mô phỏng tài khoản thật theo strategy tốt nhất: vốn ban đầu + lệnh tối thiểu mỗi vị thế + DCA, so với gửi earn ---
sim_metric = "sortino"      # chỉ số chọn strategy (giống show_best)
init_balance = 400.0        # Vốn ban đầu (USDT ~ USD)
min_order = None           # None = lệnh tối thiểu theo từng mã (minOrderAmount Bitget: 1-20$); số = áp 1 mức chung ($). Chênh lệch target - hiện tại nhỏ hơn mức này thì KHÔNG giao dịch
dca_amount = 50.0           # Số tiền nạp thêm mỗi kỳ DCA ($); 0 = tắt DCA
dca_days = 30               # Chu kỳ DCA (ngày). Mỗi kỳ nạp xong rebalance ngay về target hiện hành
earn_apr = 0.07             # Benchmark gửi earn: lãi suất danh nghĩa/năm, lãi kép theo ngày (mỗi ngày nhân 1 + apr/365), cùng vốn đầu + cùng dòng DCA. 0 = tắt

best, values, fees, done, skipped, invested = simulate_account(sim_metric, init_balance, min_order, dca_amount, dca_days)
eq_sim = values.sum(axis=1)
# equity lý tưởng: cùng dòng tiền DCA nhưng weight liên tục (không giới hạn lệnh tối thiểu), net of fees như backtest
r_net, dep = returns_wf[best].dropna().to_numpy(), invested.diff().fillna(0).to_numpy()
eq_ideal, e = np.empty(len(r_net)), init_balance
for i, ri in enumerate(r_net):
    e = (e + dep[i]) * (1 + ri); eq_ideal[i] = e
eq_ideal = pd.Series(eq_ideal, index=invested.index)
cpd = max(1, round(86400 / tf_delta(timeframe).total_seconds()))     # số candle / ngày
eq_earn, e = np.empty(len(dep)), init_balance
for i in range(len(dep)):
    e += dep[i]
    if i > 0 and i % cpd == 0:
        e *= 1 + earn_apr / 365                                       # trả lãi cuối mỗi ngày, nhập gốc
    eq_earn[i] = e
eq_earn = pd.Series(eq_earn, index=invested.index)
n_dca = int((dep > 0).sum())
print(f"Best {sim_metric}: {best} | vốn {init_balance:.0f}$ + DCA {dca_amount:.0f}$/{dca_days} ngày ({n_dca} lần), lệnh tối thiểu {'theo mã' if min_order is None else f'{min_order:.0f}$'}, phí {fee:.2%}")
print(f"Tổng vốn nạp: {invested.iloc[-1]:.2f}$ | Số dư cuối: {eq_sim.iloc[-1]:.2f}$ | lãi/lỗ: {eq_sim.iloc[-1] - invested.iloc[-1]:+.2f}$ ({eq_sim.iloc[-1] / invested.iloc[-1] - 1:+.2%})")
print(f"Lý tưởng (weight liên tục): {eq_ideal.iloc[-1]:.2f}$ ({eq_ideal.iloc[-1] / invested.iloc[-1] - 1:+.2%} trên vốn nạp)")
print(f"Gửi earn {earn_apr:.1%}/năm (kép ngày): {eq_earn.iloc[-1]:.2f}$ ({eq_earn.iloc[-1] - invested.iloc[-1]:+.2f}$, {eq_earn.iloc[-1] / invested.iloc[-1] - 1:+.2%}) | strategy hơn earn: {eq_sim.iloc[-1] - eq_earn.iloc[-1]:+.2f}$")
print(f"Lệnh đã khớp: {done} | bị bỏ vì dưới min order: {skipped} | tổng phí: {fees:.2f}$ | tiền mặt: {values['CASH'].iloc[-1]:.2f}$")
pos_usd = values.iloc[-1][lambda s: s > 0.01].sort_values(ascending=False)
print("Vị thế hiện tại:"); print(pd.DataFrame({"USD": pos_usd.round(2), "%": (pos_usd / eq_sim.iloc[-1] * 100).round(2)}).to_string())
fig = plot_stack(values, f"Equity & weights ($) - {best}")
fig.add_trace(go.Scatter(x=eq_ideal.index, y=eq_ideal, name="Equity lý tưởng (weight liên tục)", line=dict(color="black", width=1, dash="dot")))
fig.add_trace(go.Scatter(x=eq_earn.index, y=eq_earn, name=f"Gửi earn {earn_apr:.1%}/năm", line=dict(color="purple", width=1.5)))
fig.add_trace(go.Scatter(x=invested.index, y=invested, name="Tổng vốn nạp", line=dict(color="grey", width=1.5, dash="dash", shape="hv")))
fig.show()
plot_weights(weights_hist[best], f"Target weights (làm tròn theo weight_step) - {best}").show()

In [ ]:
end_time = time.time()
elapsed_seconds = end_time - start_time

# Chuyển sang định dạng hh:mm:ss
hours, rem = divmod(elapsed_seconds, 3600)
minutes, seconds = divmod(rem, 60)

print(f"Thời gian xử lý: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")